# Silver Layer — Cleaned & Standardised

Read from `bronze_vehicle_registrations`, clean and standardise, then save as `silver_vehicle_registrations`.

- **Source:** `bronze_vehicle_registrations` Delta table
- **Output:** `silver_vehicle_registrations` Delta table
- **Rules:** Cast types, rename to snake_case, drop nulls on critical fields, add year/month. No aggregations.


In [1]:
from pyspark.sql import functions as F

BRONZE_TABLE = "bronze_vehicle_registrations"
SILVER_TABLE = "silver_vehicle_registrations"

## Read from Bronze


In [2]:
df_bronze = spark.table(BRONZE_TABLE)

print(f"Bronze row count: {df_bronze.count()}")
df_bronze.printSchema()

Bronze row count: 938322
root
 |-- date_reg: date (nullable = true)
 |-- type: string (nullable = true)
 |-- maker: string (nullable = true)
 |-- model: string (nullable = true)
 |-- colour: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



## Clean and standardise


In [3]:
# Rename columns to meaningful snake_case names, cast types, and add year/month
df_silver = (
    df_bronze
    .withColumnRenamed("date_reg", "registration_date")
    .withColumnRenamed("type", "vehicle_type")
    .withColumnRenamed("maker", "manufacturer")
    .withColumnRenamed("model", "model")
    .withColumnRenamed("fuel", "fuel_type")
    .withColumnRenamed("state", "state")
    .withColumn("year", F.year("registration_date"))
    .withColumn("month", F.month("registration_date"))
    .drop("ingested_at")
)

## Drop rows with null critical fields


In [4]:
# Drop rows where essential fields are missing
critical_cols = ["registration_date", "manufacturer", "state"]
df_silver = df_silver.dropna(subset=critical_cols)

print(f"Rows after null drop: {df_silver.count()}")

Rows after null drop: 938322


## Write to Delta table


In [5]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

print(f"Written to table: {SILVER_TABLE}")

Written to table: silver_vehicle_registrations


## Verify the table


In [6]:
df_check = spark.table(SILVER_TABLE)

print(f"Row count: {df_check.count()}")
print(f"Columns: {df_check.columns}")
df_check.printSchema()
df_check.show(5)

Row count: 938322
Columns: ['registration_date', 'vehicle_type', 'manufacturer', 'model', 'colour', 'fuel_type', 'state', 'year', 'month']
root
 |-- registration_date: date (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- colour: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+-----------------+------------+------------+---------+------+-----------+-----------+----+-----+
|registration_date|vehicle_type|manufacturer|    model|colour|  fuel_type|      state|year|month|
+-----------------+------------+------------+---------+------+-----------+-----------+----+-----+
|       2025-01-01|     motokar|         BYD|     Seal| white|   electric|Rakan Niaga|2025|    1|
|       2025-01-01|  window_van|         Cam| Placer-X|yellow|greendiesel|      Johor|2025|    1|
| 